In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Descargar cierres SAN.MC último año
end = datetime.now()
start = end - timedelta(days=365)
ams = yf.download('TSLA', start=start, end=end)['Close'].reset_index()
ams.columns = ['Date', 'Close_TSLA']
ams['Date'] = pd.to_datetime(ams['Date']).dt.date
ams.to_csv('TSLA_cierres.csv', index=False)
print("✅ TSLA_cierres.csv guardado!")
print(ams.tail())

KeyboardInterrupt: 

In [3]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta


# 1. DATOS SAN.MC (1 año)
end = datetime.now()
start = end - timedelta(days=365)
df = yf.download('TSLA', start=start, end=end)['Close'].reset_index()
df.columns = ['Date', 'Close']
df.to_csv('TSLA_data.csv', index=False)


# 2. BACKTEST CORREGIDO
df['Return'] = df['Close'].pct_change() * 100
capital = 10000
position = 0  # 0 = cash, 1 = comprado
buy_price = 0
buy_day = 0
operaciones = []


for i in range(1, len(df)):
    today_price = df['Close'].iloc[i]
    today_ret = df['Return'].iloc[i]
   
    # SEÑAL COMPRA: -5% y sin posición
    if today_ret <= -5 and position == 0:
        position = 1
        buy_price = df['Close'].iloc[i]  # Precio cierre día caída
        buy_day = i
        capital_invertido = capital
        capital = 0
        operaciones.append(['BUY', df['Date'].iloc[i], buy_price, capital_invertido])
   
    # SI COMPRADO: check día SIGUIENTE (i = buy_day + 1)
    elif position == 1 and i == buy_day + 1:
        next_day_ret = (today_price / buy_price - 1) * 100
       
        if next_day_ret >= 3:  # +2% → VENTA GANADORA
            capital = capital_invertido * (today_price / buy_price)
            operaciones.append(['SELL +2%', df['Date'].iloc[i], today_price, capital])
            position = 0
        else:  # NO +2% → VENTA al cierre día siguiente
            today_price = df['Close'].iloc[i+7]  # Precio cierre día siguiente
            capital = capital_invertido * (today_price / buy_price)
            operaciones.append(['SELL cierre', df['Date'].iloc[i+7], today_price, capital])
            position = 0


# Posición final
if position == 1:
    capital = capital_invertido * (df['Close'].iloc[-1] / buy_price)


print(f"💰 INICIAL: €10.000")
print(f"💵 FINAL: €{capital:,.0f}")
print(f"📈 RENDIMIENTO: {(capital/10000-1)*100:+.1f}%")
print(f"🔢 OPERACIONES: {len(operaciones)}")


pd.DataFrame(operaciones, columns=['Tipo', 'Fecha', 'Precio', 'Capital']).to_csv('TSLA_backtest_corregido.csv', index=False)
df.to_csv('TSLA_completo.csv', index=False)


print("\n✅ Archivos:")

[*********************100%***********************]  1 of 1 completed

💰 INICIAL: €10.000
💵 FINAL: €9,415
📈 RENDIMIENTO: -5.9%
🔢 OPERACIONES: 32

✅ Archivos:


In [1]:
#miguel es tonto